**Install Required Packages**

In [1]:
!pip install google-cloud-bigquery google-cloud-storage google-auth


**Import libraries and initialize client instances**


In [2]:
import os
import asyncio
import google.auth

os.environ['GOOGLE_CLOUD_PROJECT']

GOOGLE_CLOUD_PROJECT = "us-gcp-ame-con-e5556-npd-1"
GOOGLE_CLOUD_LOCATION = "us-central1"
MODEL_NAME = "gemini-2.5-flash"

os.environ['GOOGLE_CLOUD_PROJECT'] = GOOGLE_CLOUD_PROJECT
os.environ['GOOGLE_CLOUD_LOCATION'] = GOOGLE_CLOUD_LOCATION
os.environ['MODEL_NAME'] = MODEL_NAME

google.auth.default()

(<google.auth.compute_engine.credentials.Credentials at 0x7aa3f02f99a0>,
 'us-gcp-ame-con-e5556-npd-1')

In [3]:
os.getenv("GOOGLE_CLOUD_PROJECT")
os.getenv("GOOGLE_CLOUD_LOCATION")

'us-central1'

In [17]:
from google.cloud import bigquery
from google.cloud import storage

# Initialize BigQuery and Storage clients
bigquery_client = bigquery.Client()
storage_client = storage.Client()

# --- Configuration for your data load ---
# Replace with your project ID, dataset ID, table ID, and GCS bucket details
project_id = "us-gcp-ame-con-e5556-npd-1"
location = "us-central1"
dataset_id = "emergency_response"
raw_table_id = "weather_data_raw"
gold_table_id = "weather_commnication_data"
model_id = "public_communication_model"
source_bucket_name = "gs://labs.roitraining.com/data-to-ai-workshop"
source_file_name = "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv"

**Create Dataset**

In [5]:
sql_ddl_dataset_statement = f"""
CREATE SCHEMA IF NOT EXISTS `{project_id}.{dataset_id}`
OPTIONS(
    location = '{location}',
    default_table_expiration_days=14
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_dataset_statement)

# Wait for the job to complete
query_job.result()

print(f"Dataset '{dataset_id}' created or already exists in project '{project_id}' at location '{location}'.")


Dataset 'emergency_response' created or already exists in project 'us-gcp-ame-con-e5556-npd-1' at location 'us-central1'.


**Create Table**

In [26]:
sql_ddl_statement = f"""
DROP TABLE IF EXISTS `{project_id}.{dataset_id}.{raw_table_id}`;

CREATE TABLE `{project_id}.{dataset_id}.{raw_table_id}` (
    date DATE,
    city STRING,
    state STRING,
    temperature_f NUMERIC,
    wind_speed_mph NUMERIC,
    precipitation_in NUMERIC,
    barometric_pressure_inHg NUMERIC,
    humidity_percent NUMERIC,
    weather_condition STRING
    );
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' created or replaced in dataset '{dataset_id}'.")


Table 'weather_data_raw' created or replaced in dataset 'emergency_response'.


**Load data into Big Query Table**

In [27]:
sql_ddl_statement = f"""
LOAD DATA INTO `{project_id}.{dataset_id}.{raw_table_id}`
(
    date DATE,
    city STRING,
    state STRING,
    temperature_f NUMERIC,
    wind_speed_mph NUMERIC,
    precipitation_in NUMERIC,
    barometric_pressure_inHg NUMERIC,
    humidity_percent NUMERIC,
    weather_condition STRING
)
FROM FILES (
format = 'CSV',
field_delimiter = ',',
max_bad_records = 10,
uris = ['{source_file_name}']);
"""
# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_ddl_statement)

# Wait for the job to complete
query_job.result()

print(f"Table '{raw_table_id}' loaded with raw data from '{source_file_name}'.")

Table 'weather_data_raw' loaded with raw data from 'gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv'.


In [28]:
df = bigquery_client.list_rows(f"{project_id}.{dataset_id}.{raw_table_id}").to_dataframe()
df


,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.700000000,5.000000000,0.120000000,29.800000000,50.400000000,Cloudy
1,2025-02-26,Atlanta,GA,75.200000000,10.400000000,0.030000000,29.580000000,49.900000000,Cloudy
2,2025-03-01,Atlanta,GA,51.700000000,4.700000000,0.080000000,29.740000000,49.900000000,Cloudy
3,2025-03-05,Atlanta,GA,74.400000000,5.100000000,0.020000000,29.920000000,50.400000000,Cloudy
4,2025-03-10,Atlanta,GA,59.500000000,9.600000000,0.090000000,29.670000000,57.200000000,Cloudy
...,...,...,...,...,...,...,...,...,...
295,2025-03-19,San Francisco,CA,75.100000000,3.600000000,0.020000000,29.850000000,27.000000000,Sunny
296,2025-02-19,Seattle,WA,92.800000000,0.900000000,0.010000000,29.970000000,31.700000000,Sunny
297,2025-02-22,Seattle,WA,73.700000000,3.600000000,0.020000000,30.160000000,32.600000000,Sunny
298,2025-03-01,Seattle,WA,87.900000000,5.800000000,0.040000000,30.230000000,24.000000000,Sunny


**Create Connection to Gemini**

**Create ML Model using Gemini LLM**

In [30]:
create_remote_model_statement = f"""
CREATE OR REPLACE MODEL `{project_id}.{dataset_id}.{model_id}`
REMOTE WITH CONNECTION `{location}.gemini_connection`
OPTIONS (
  ENDPOINT = 'gemini-2.5-flash'
);
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(create_remote_model_statement)

# Wait for the job to complete
query_job.result()

print(f"Remote model '{model_id}' created or replaced in dataset '{dataset_id}'.")

Remote model 'public_communication_model' created or replaced in dataset 'emergency_response'.


**Create Weather Report for each record**

In [33]:
sql_query_statement = f"""
CREATE OR REPLACE TABLE `{project_id}.{dataset_id}.{gold_table_id}` AS
SELECT JSON_VALUE(ml_generate_text_result,'$.candidates[0].content.parts[0].text') as weather_report,
  * Except(prompt,ml_generate_text_status, ml_generate_text_result)
FROM
  ML.GENERATE_TEXT(MODEL `{project_id}.{dataset_id}.{model_id}`,
    (
      SELECT
        *,
        Concat('generate a weather report or warning based on the data provided. City: ' , city, ' State: ', state, ' Temperature: ', temperature_f, ' Humidity: ', humidity_percent, ' Barometric Pressure: ', barometric_pressure_inHg, ' Precipitation: ', precipitation_in, ' Wind Speed: ', wind_speed_mph, ' Weather Condition: ', weather_condition ) AS prompt
      FROM
        `{project_id}.{dataset_id}.{raw_table_id}`
    ),
    STRUCT(
      0.2 AS temperature,
      0.9 AS top_p,
      1000 AS max_output_tokens
    )
  );
"""

# Execute the DDL statement using the BigQuery client
query_job = bigquery_client.query(sql_query_statement)

# Wait for the job to complete
query_job.result()

print(f"Weather report generated and stored in table '{gold_table_id}'.")
df_gold = bigquery_client.list_rows(f"{project_id}.{dataset_id}.{gold_table_id}").to_dataframe()
df_gold

Weather report generated and stored in table 'weather_commnication_data'.


,weather_report,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,"Here's a weather report for Atlanta, GA based ...",2025-02-26,Atlanta,GA,75.200000000,10.400000000,0.030000000,29.580000000,49.900000000,Cloudy
1,"Here's a weather report for Atlanta, GA, based...",2025-03-10,Atlanta,GA,59.500000000,9.600000000,0.090000000,29.670000000,57.200000000,Cloudy
2,"**Weather Report for Atlanta, Georgia**\n\n**C...",2025-03-05,Atlanta,GA,74.400000000,5.100000000,0.020000000,29.920000000,50.400000000,Cloudy
3,"**Weather Report for Atlanta, Georgia**\n\n**C...",2025-02-21,Atlanta,GA,55.700000000,5.000000000,0.120000000,29.800000000,50.400000000,Cloudy
4,"**Weather Report for Atlanta, Georgia**\n\n**C...",2025-03-14,Atlanta,GA,71.700000000,7.200000000,0.180000000,29.920000000,55.300000000,Cloudy
...,...,...,...,...,...,...,...,...,...,...
295,"**Weather Report: San Francisco, CA**\n\n**Cur...",2025-02-19,San Francisco,CA,93.800000000,3.100000000,0.040000000,30.030000000,20.800000000,Sunny
296,"**Weather Report for Seattle, Washington**\n\n...",2025-03-01,Seattle,WA,87.900000000,5.800000000,0.040000000,30.230000000,24.000000000,Sunny
297,"**Weather Report: Seattle, Washington**\n\nGoo...",2025-03-19,Seattle,WA,74.800000000,2.100000000,0.020000000,30.240000000,25.800000000,Sunny
298,"**Weather Report: Seattle, WA**\n\n**Current C...",2025-02-19,Seattle,WA,92.800000000,0.900000000,0.010000000,29.970000000,31.700000000,Sunny
